# 2025 서울 상권 유동인구 대비 매출성과 분석

> Open API → GCS → BigQuery → dbt → Python → Looker Studio

서울시 상권분석 데이터를 이용해 2025년 상권별 상대적 매출성과를 비교하고, 성과 차이가 거래 발생 강도·평균 결제금액·상업 기반·업종 구성·입지 특성과 어떤 관계를 보이는지 분석했습니다.

이 노트북은 포트폴리오 열람을 위한 요약본입니다. 데이터 품질 검증과 전체 분석 코드는 [seoul_commercial_analysis.ipynb](seoul_commercial_analysis.ipynb)에서 확인할 수 있습니다.

## 1. 문제 정의

### 분석 질문

1. 유동인구 대비 매출성과가 높은 상권과 낮은 상권은 어디인가?
2. 성과 차이는 거래 발생 강도와 평균 결제금액 중 무엇과 더 관련되는가?
3. 점포·직장인구·집객시설·업종 구성은 성과와 어떤 관계를 보이는가?
4. 주변 상권의 밀집과 지역적 위치도 성과 차이를 설명하는가?

### 핵심 해석 원칙

`매출/유동인구`와 `거래건수/유동인구`는 실제 개인별 구매전환율이 아닙니다. 유동인구는 고유 방문자 수가 아닌 추정 유동량이고, 매출건수도 고유 구매자 수가 아니므로 상대적 성과와 거래 발생 강도로 해석합니다.

## 2. 데이터 파이프라인

```mermaid
flowchart LR
    A[서울 열린데이터광장<br/>Open API] --> B[CSV]
    B --> C[GCS<br/>Raw 보관]
    C --> D[BigQuery<br/>Raw tables]
    D --> E[dbt Staging<br/>정제·표준화]
    E --> F[dbt Mart<br/>분기 × 상권]
    F --> G[Python<br/>분석]
    G --> H[Looker Studio<br/>대시보드]
```

### 데이터 모델링

- Staging: 컬럼명 표준화, 타입 변환, 2025년 필터링
- Mart: 매출·점포 집계 후 유동인구·직장인구·집객시설·상권정보 결합
- Grain: `분기 × 상권`
- Key: `trade_area_quarter_key`
- Test: `unique`, `not_null`, `accepted_values`

## 3. 데이터와 분석 대상

| 데이터 | 주요 변수 |
|---|---|
| 추정매출 | 매출액, 매출건수, 요일·시간대별 매출 |
| 점포 | 전체·프랜차이즈·개폐업 점포 수 |
| 유동인구 | 전체·요일·시간대별 추정 유동인구 |
| 직장인구 | 전체·성별 직장인구 |
| 집객시설 | 교통·문화·의료·교육·공공·숙박시설 |
| 상권영역 | 상권명, 유형, 좌표, 면적, 행정구역 |
| 상권변화지표 | 운영·폐업 기간, 변화 유형 |

- 분석 기간: 2025년 1~4분기
- 원본 Mart: 6,276행
- 연간 전체 상권: 1,577개
- 최종 분석 대상: 1,558개
- 선정 조건: 매출·매출건수·유동인구가 네 분기 모두 존재하고 각 합계가 0보다 큼

## 4. 핵심 KPI

### 유동인구 대비 매출성과

```text
sales_per_flow = 연간 총매출 ÷ 연간 총유동인구
```

### 성과 분해

```text
유동인구 대비 매출
= 유동인구 대비 거래건수 × 평균 결제금액
```

```text
transactions_per_flow = 총매출건수 ÷ 총유동인구
average_transaction_value = 총매출 ÷ 총매출건수
```

### 업종 집중도

```text
HHI = Σ(업종별 매출 비중²)
```

HHI가 낮을수록 매출이 여러 업종에 분산되고, 높을수록 특정 업종에 집중됩니다.

## 5. 서울 전체 Top 5·Bottom 5

### Top 5

| 순위 | 상권 | 유형 | 매출/유동인구 |
|---:|---|---|---:|
| 1 | 가락시장 | 발달상권 | 14,064,348 |
| 2 | 강서농산물도매시장 | 발달상권 | 9,947,162 |
| 3 | 동서시장 | 전통시장 | 1,940,637 |
| 4 | 경동시장 | 전통시장 | 1,811,699 |
| 5 | 인왕시장(홍제골목형상점가) | 전통시장 | 1,281,692 |

Top 5는 시장·도매형 상권이 대부분이었습니다. 측정 유동인구에 비해 총매출과 거래건수가 매우 크고 특정 핵심 업종의 비중이 높은 전문화 상권이라는 공통점이 나타났습니다.

### Bottom 5

| 순위 | 상권 | 유형 | 매출/유동인구 |
|---:|---|---|---:|
| 1 | 신목동역 | 골목상권 | 2.65 |
| 2 | 강북문화정보도서관 | 골목상권 | 4.81 |
| 3 | 종로5가역 4번 | 골목상권 | 9.35 |
| 4 | 충정로역 7번 | 골목상권 | 9.46 |
| 5 | 연희지하차도 | 골목상권 | 11.22 |

Bottom 5에는 점포와 매출 제공 업종이 적은 소규모 상권이 포함되어 있어, 낮은 비율을 상권 전체의 구매전환 실패로 일반화하지 않았습니다.

## 6. 무엇이 성과 차이와 관련됐는가?

전체 1,558개 상권에서 Spearman 순위상관계수를 계산했습니다.

| 변수 | 총매출과의 ρ | 매출/유동인구와의 ρ | 거래건수/유동인구와의 ρ |
|---|---:|---:|---:|
| 총매출건수 | 0.946 | 0.673 | 0.782 |
| 평균 점포 수 | 0.904 | 0.567 | 0.613 |
| 매출 업종 수 | 0.858 | 0.498 | 0.560 |
| 집객시설 수 | 0.733 | 0.410 | 0.474 |
| 직장인구 | 0.584 | 0.387 | 0.354 |
| 업종 HHI | -0.628 | -0.340 | -0.450 |

### 해석

- 매출성과 차이는 평균 결제금액보다 측정 유동인구 대비 거래건수의 차이와 더 밀접했습니다.
- 점포·거래·업종 규모가 큰 상권일수록 총매출과 상대 성과가 높은 경향이 있었습니다.
- 전체적으로 업종 매출이 분산된 상권이 더 높은 성과를 보였지만, 도매·전통시장에는 집중도와 성과가 모두 높은 예외가 존재했습니다.
- 상관계수는 연관성의 방향과 크기를 나타낼 뿐 인과효과를 의미하지 않습니다.

## 7. 강남 마이스 관광특구 심층 사례

| 지표 | 결과 |
|---|---:|
| 서울 전체 연간 순위 | 9위 |
| 관광특구 내부 순위 | 1위 |
| 다업종 분산형 상권 순위 | 1위 |
| 매출/유동인구 | 901,168 |
| 거래건수/유동인구 | 30.32 |
| 매출 업종 수 | 30개 |
| 1위 업종 매출 비중 | 21.2% |
| 상위 3개 업종 매출 비중 | 44.0% |
| 업종 HHI | 0.093 |

강남 마이스는 특정 업종에 매출이 집중된 시장형 Top 상권과 달리, 여러 업종에 매출이 분산되면서도 높은 거래 발생 강도를 보였습니다. 또한 네 분기 모두 서울 전체 상위 10위권을 유지해 일시적인 비율 상승이 아닌 안정적인 고성과 사례로 판단했습니다.

## 8. 공간·입지 분석

### 지역별 차이

| 자치구 | 매출/유동인구 중앙값 |
|---|---:|
| 중구 | 22,712 |
| 강남구 | 18,311 |
| 서초구 | 13,799 |

행정동에서는 중구 소공동 약 72,634, 강서구 가양1동 약 63,276, 중구 회현동 약 54,526 순으로 높았습니다. 다만 지역마다 상권 수와 유형 구성이 다르므로 이를 입지의 인과효과로 해석하지 않았습니다.

### 주변 상권과 성과

| 공간 변수 | 매출/유동인구와의 Spearman ρ |
|---|---:|
| 500m 이내 상권 수 | 0.029 |
| 1km 이내 상권 수 | -0.048 |
| 500m 이웃 평균 성과 | 0.157 |
| 500m 이웃 평균 직장인구 | 0.130 |

동대문·동묘·청량리에서 상권 밀집이 확인됐지만, 주변 상권이 많다는 사실만으로 개별 상권의 성과가 높아지지는 않았습니다. 현재 데이터에서는 위치 자체보다 점포 수·거래건수·업종 구성 같은 상권 내부 구조가 성과와 더 밀접하게 연결됐습니다.

## 9. 탐색적 공간 패턴

각 상권과 1km 이내 이웃 상권의 성과를 서울 전체 중앙값과 비교했습니다.

| 패턴 | 상권 수 | 의미 |
|---|---:|---|
| High–High | 702 | 해당 상권과 주변 평균이 모두 높음 |
| Low–High | 689 | 해당 상권은 낮지만 주변 평균은 높음 |
| Low–Low | 86 | 해당 상권과 주변 평균이 모두 낮음 |
| High–Low | 71 | 해당 상권은 높지만 주변 평균은 낮음 |
| No neighbor | 10 | 1km 안에 비교 상권이 없음 |

High–Low에는 마들역, 구로역, 수유전통시장 등이 포함됐습니다. 이 분류는 Local Moran’s I나 p-value를 사용한 통계적 공간 군집 검정이 아니라 후속 분석 후보를 찾기 위한 탐색적 분류입니다.

## 10. Looker Studio 대시보드

**[대시보드에서 분석 결과 살펴보기](https://datastudio.google.com/reporting/d8b2094c-6cb4-46ab-b319-6d493e19e47f)**

분석 결과를 비개발자도 탐색할 수 있도록 한 화면의 대시보드로 구성했습니다.

- 자치구·상권 유형·성과 유형·상권명 필터
- 분석 대상 상권 수와 전체 성과 1위 KPI
- 다업종 분산형 성과 1위 KPI
- 유동인구 규모와 매출성과 분포 산점도
- 전체 Top 5·Bottom 5
- 관광특구 비교
- 강남 마이스 거래 발생 강도·업종 수·HHI 요약

대시보드에서 자치구·상권 유형·성과 유형·상권명을 직접 선택해 전체 분포와 개별 상권의 성과를 비교할 수 있습니다.

## 11. 핵심 인사이트

1. **전체 Top 5는 일반 상권이 아니라 시장·도매형 특수 상권이었다.**  
   전체 순위를 그대로 구매전환율로 해석하면 데이터 포착 범위 차이를 놓칠 수 있습니다.

2. **성과 차이는 객단가보다 거래 발생 강도와 더 밀접했다.**  
   높은 성과는 주로 측정 유동인구 대비 많은 거래건수에서 나타났습니다.

3. **상권의 상업 기반이 중요했다.**  
   점포 수, 매출건수, 업종 수는 매출 및 상대 성과와 비교적 강한 관계를 보였습니다.

4. **단순 상권 밀집도만으로는 성과를 설명하기 어려웠다.**  
   반경 내 상권 수와 성과의 관계는 거의 없었고, 주변 평균 특성과의 관계도 약했습니다.

5. **강남 마이스는 구조적으로 다른 고성과 사례였다.**  
   업종이 분산되어 있으면서 높은 거래 발생 강도와 분기별 안정성을 함께 보였습니다.

## 12. 분석 한계

- 유동인구는 고유 방문자 수가 아닌 추정 유동량입니다.
- 매출건수는 고유 구매자 수가 아니므로 실제 구매전환율을 계산할 수 없습니다.
- 추정매출과 유동인구의 공간·업종 포착 범위가 다를 수 있습니다.
- 업종 수와 집중도는 매출 데이터에 나타난 업종을 기준으로 합니다.
- 상권 간 거리는 중심점 간 직선거리이며 실제 보행거리가 아닙니다.
- 공간 패턴은 통계적 유의성 검정을 포함하지 않은 탐색적 분류입니다.
- 모든 결과는 변수 간 연관성을 보여줄 뿐 인과효과를 증명하지 않습니다.

## 13. 후속 분석

- 지하철역 거리와 역별 승하차 인원 결합
- 도로망 기반 보행거리와 상권 폴리곤 분석
- 상권 유형·규모를 통제한 기대 매출 및 초과성과 분석
- 거래 발생 강도와 평균 결제금액을 이용한 성과 유형화
- 주문·고객 단위 데이터를 이용한 실제 전환율·리텐션 분석

## 14. 기술 스택과 산출물

| 영역 | 기술 |
|---|---|
| 수집 | Python, 서울 열린데이터광장 Open API |
| 저장 | Google Cloud Storage |
| 웨어하우스 | Google BigQuery |
| 모델링 | dbt |
| 분석 | Python, Pandas, NumPy |
| 시각화 | Looker Studio |

### 주요 산출물

- [전체 분석 노트북](seoul_commercial_analysis.ipynb)
- [프로젝트 README](../README.md)
- [`mart_trade_area_quarterly` 모델](../models/marts/mart_trade_area_quarterly.sql)
- [dbt 테스트 정의](../models/marts/schema.yml)